In [3]:
from cmdstanpy import CmdStanModel
import numpy as np
import pandas as pd
from pathlib import Path
import scipy.stats as st
import os

from joblib import Parallel, delayed
from tqdm.auto import tqdm
import traceback
import ast

from __future__ import annotations
from dataclasses import dataclass
from typing import Any, Dict, Optional

#### Compile Stan model

In [5]:
stan_path = Path("/Users/katsiarynadavydzenka/Documents/PhD_AI/DeConveil/stan/differential_dosage_model_v1.stan")
model = CmdStanModel(stan_file=str(stan_path))

#### Fit one gene / multiple genes - CRC dataset

In [5]:
DATA_PATH = "/Users/katsiarynadavydzenka/Documents/PhD_AI/CRC_case_study/data/stan_model_test/"
df_long = pd.read_csv(os.path.join(DATA_PATH, "crc_joint_long_de_drivers.csv"))
df_long.head()

,gene,sampleID,expr,copies,subtype,purity,ploidy,sf
0,ACVR1B,CRC.SW.U0001.T,7593,2,MSS,0.41,3.8,1.293201
1,ACVR1B,CRC.SW.U0002.T,5003,2,MSS,0.37,3.3,1.024008
2,ACVR1B,CRC.SW.U0004.T,4994,2,MSS,0.58,3.2,1.169870
3,ACVR1B,CRC.SW.U0030.T,6231,2,MSS,0.59,2.2,1.290047
4,ACVR1B,CRC.SW.U0066.T,2908,2,MSS,0.41,1.9,0.971163


In [15]:
# Select one gene
gene_df = df_long[df_long["gene"] == "BRAF"]
gene_df.head()

,gene,sampleID,expr,copies,subtype,purity,ploidy,sf
17240,BRAF,CRC.SW.U0001.T,3100,2,MSS,0.41,3.8,1.293201
17241,BRAF,CRC.SW.U0002.T,2913,3,MSS,0.37,3.3,1.024008
17242,BRAF,CRC.SW.U0004.T,2028,2,MSS,0.58,3.2,1.169870
17243,BRAF,CRC.SW.U0030.T,2008,2,MSS,0.59,2.2,1.290047
17244,BRAF,CRC.SW.U0066.T,2066,2,MSS,0.41,1.9,0.971163


In [9]:
# Select multiple genes
#gene_df = df_long[df_long["gene"].isin(["BRAF", "KRAS", "PIK3CA", "APC", "SMAD4"])]
#gene_df.head()

#### Model Simulator

In [17]:
# simulate covariates

def simulate_covariates(
    N=200,
    subtype_levels=("A", "B"),
    seed=123,
):
    """Simulate sample-level covariates shared by all genes."""
    
    rng = np.random.default_rng(seed)
    S = len(subtype_levels)

    # Subtypes: roughly balanced
    subtype_idx = rng.integers(low=0, high=S, size=N)  # 0..S-1
    subtype_labels = np.array(subtype_levels)[subtype_idx]

    # Library size factors (around 1)
    sf = rng.lognormal(mean=0.0, sigma=0.2, size=N)

    # Tumor purity
    purity = rng.beta(a=5.0, b=2.0, size=N)  # mean ~0.7

    # Copy number per sample
    copies = rng.choice([1, 2, 3, 4, 5], size=N, p=[0.15, 0.5, 0.15, 0.10, 0.10]).astype(float)

    CN_eff = np.clip(copies, 1.0, None)
    dose_log = np.log(CN_eff / 2.0)
    dev = (copies - 2.0) / 2.0

    covars = {
        "N": N,
        "S": S,
        "subtype_levels": subtype_levels,
        "subtype_idx": subtype_idx,
        "subtype_labels": subtype_labels,
        "sf": sf,
        "purity": purity,
        "copies": copies,
        "dose_log": dose_log,
        "dev": dev,
    }
    return covars

# Simulate dispersion
#def simulate_phi(rng, center=5.0, log_sd=0.8, lo=0.5, hi=20.0):
    #"""
    #Simulate a realistic NB2 dispersion parameter phi for one gene.
    #center: typical value (median of the lognormal)
    #log_sd: spread on the log scale
    #lo, hi: clamp phi into [lo, hi] to avoid wild extremes
    #"""
    #log_phi = rng.normal(np.log(center), log_sd)
    #phi = float(np.exp(log_phi))
    # clamp into reasonable range
    #phi = max(lo, min(phi, hi))
    #return phi


# simulate true parameters for one gene

def simulate_gene_params(
    covars,
    rng,
    gene_id,
):
    """Simulate true parameters for one gene."""
    S = covars["S"]

    # Global baselines: log mean at CN=2, sf=1, purity=1
    # Let gene-level baseline vary around log(20) with some spread
    b0_mean = rng.normal(np.log(200.0), 0.5)
    b0_offset = rng.normal(0.0, 0.3, size=S)
    b0_off = b0_offset - b0_offset.mean()
    b0 = b0_mean + b0_off

    # Proportional dosage sensitivity
    b_scaling_mean = rng.normal(0.6, 0.3)           # around 0.6
    b_scaling_offset = rng.normal(0.0, 0.3, size=S)
    b_scaling_off = b_scaling_offset - b_scaling_offset.mean()
    b_scaling = b_scaling_mean + b_scaling_off

    # Deviation from scaling (compensation / over-scaling)
    b_dev_mean = rng.normal(0.0, 0.15)              # centered near 0
    b_dev_offset = rng.normal(0.0, 0.1, size=S)
    b_dev_off = b_dev_offset - b_dev_offset.mean()
    b_deviation = b_dev_mean + b_dev_off

    # Non-tumor baseline (stroma)
    b_noncancer_log = rng.normal(np.log(5.0), 0.3)

    # Dispersion (per gene)
    # lognormal centered around phi ~ 5
    
    log_phi = rng.normal(np.log(5.0), 0.5)
    phi = float(np.exp(log_phi))

    true_params = {
        "gene": gene_id,
        "b0": b0,
        "b_scaling": b_scaling,
        "b_deviation": b_deviation,
        "b_noncancer_log": b_noncancer_log,
        "phi": phi,
        "b0_mean": b0_mean,
        "b_scaling_mean": b_scaling_mean,
        "b_dev_mean": b_dev_mean,
    }
    return true_params

# simulate counts for one gene given covariates + params

def simulate_counts_for_gene(covars, true_params, rng):
    """Simulate NB2 counts for one gene, given covariates and true parameters."""
    N = covars["N"]
    S = covars["S"]
    subtype_idx = covars["subtype_idx"]
    sf = covars["sf"]
    purity = covars["purity"]
    copies = covars["copies"]
    dose_log = covars["dose_log"]
    dev = covars["dev"]

    b0 = true_params["b0"]
    b_scaling = true_params["b_scaling"]
    b_deviation = true_params["b_deviation"]
    b_noncancer_log = true_params["b_noncancer_log"]
    phi = true_params["phi"]

    mu = np.empty(N)
    for n in range(N):
        s = subtype_idx[n]   # 0..S-1

        linpred = (
            b0[s]
            + dose_log[n] * b_scaling[s]
            + dev[n] * b_deviation[s]
        )

        tumor_mu = sf[n] * purity[n] * np.exp(linpred)
        stroma_mu = sf[n] * (1.0 - purity[n]) * np.exp(b_noncancer_log)
        mu[n] = tumor_mu + stroma_mu

    # Stan NB2(mu, phi) -> numpy negative_binomial(n=phi, p=phi/(phi+mu))
    r = phi
    p = r / (r + mu)
    p = np.clip(p, 1e-8, 1 - 1e-8)

    expr = rng.negative_binomial(n=r, p=p, size=N).astype(int)
    return expr

# Simulate multigene dataset
def simulate_dataset_multi_gene(
    G=100,
    N=200,
    subtype_levels=("A", "B"),
    seed=123,
):
    rng = np.random.default_rng(seed)

    # 1. Sample-level covariates shared across all genes
    covars = simulate_covariates(N=N, subtype_levels=subtype_levels, seed=seed)
    N = covars["N"]  # ensure

    all_rows = []
    truth_rows = []

    for g in range(G):
        gene_id = f"G{g+1}"

        # 2. True parameters for this gene
        true_g = simulate_gene_params(covars, rng, gene_id)

        # 3. Simulate counts
        expr = simulate_counts_for_gene(covars, true_g, rng)

        # 4. Store rows for this gene
        df_g = pd.DataFrame({
            "gene": gene_id,
            "expr": expr,
            "copies": covars["copies"],
            "purity": covars["purity"],
            "sf": covars["sf"],
            "subtype": covars["subtype_labels"],
        })
        all_rows.append(df_g)

        # 5. Store truth for this gene (flatten subtype params)
        row_truth = {
            "gene": gene_id,
            "phi_true": true_g["phi"],
            "b_noncancer_log_true": true_g["b_noncancer_log"],
            "b0_mean_true": true_g["b0_mean"],
            "b_scaling_mean_true": true_g["b_scaling_mean"],
            "b_dev_mean_true": true_g["b_dev_mean"],
        }
        for s, lvl in enumerate(covars["subtype_levels"], start=1):
            row_truth[f"b0_s{s}_true"] = true_g["b0"][s-1]
            row_truth[f"b_scaling_s{s}_true"] = true_g["b_scaling"][s-1]
            row_truth[f"b_deviation_s{s}_true"] = true_g["b_deviation"][s-1]
        truth_rows.append(row_truth)

    sim_df = pd.concat(all_rows, ignore_index=True)
    truth_df = pd.DataFrame(truth_rows)

    return sim_df, truth_df, covars

In [19]:
# Simulate 100 genes
sim_df, truth_df, covars = simulate_dataset_multi_gene(
    G=3,
    N=200,
    subtype_levels=("A", "B"),
    seed=42,
)

In [21]:
sim_df

,gene,expr,copies,purity,sf,subtype
0,G1,102,3.0,0.263995,1.083238,A
1,G1,328,5.0,0.810441,0.834355,B
2,G1,158,2.0,0.610692,0.927157,B
3,G1,116,2.0,0.602323,1.296730,A
4,G1,151,2.0,0.792361,0.931226,A
...,...,...,...,...,...,...
595,G3,100,2.0,0.636890,0.598498,A
596,G3,240,2.0,0.757525,0.953734,A
597,G3,58,3.0,0.458349,1.035933,A
598,G3,142,2.0,0.544902,1.060986,A


#### Model fit function

In [23]:
# more diagnostic parameters + MCMC + VI

def fit_one_gene_de(
    gene_df: pd.DataFrame,
    model: "CmdStanModel",
    gene: str | None = None,
    cna: str = "all",
    et: float = 0.15,
    min_aneup: int = 5,
    min_unique_counts: int = 5,
    min_cn_abs_sum: float = 1.0,
    subtype_col: str = "subtype",
    subtype_order: list[str] | None = None,
    chains: int = 4,
    iter_warmup: int = 1000,
    iter_sampling: int = 1000,
    seed: int = 1,
    show_progress: bool = False,
    adapt_delta: float = 0.9,
    max_treedepth: int = 12,
    rope_logfc: float = float(np.log(1.2)),
    eps_frac: float = 0.10,          # ROPE on fracCN for ~10% change
    return_all_subtypes: bool = True,

    # NEW: inference engine & VI settings
    engine: str = "nuts",               # "nuts", "vi_meanfield", "vi_fullrank"
    vi_iter: int = 20000,
    vi_output_samples: int = 2000,
    vi_elbo_samples: int = 100,
    vi_grad_samples: int = 1,
):
    """
    Fit the Bayesian differential gene-dosage model for a single gene.

    Required columns in gene_df:
        expr, copies, purity, sf, subtype_col
    Optional:
        gene (if gene_df contains multiple genes)

    Stan model is the log-link CN-expression model with generated quantities:
        delta_tumor0_log, delta_scaling, delta_dev,
        lp_2to1[s], lp_2to3[s], lp_2to4[s],
        (and optionally fracCN_*, cancel_index_*, p_DC_* if you added them).
    """

    df = gene_df.copy()

    # ---- subset gene ----
    if gene is not None and "gene" in df.columns and df["gene"].nunique() > 1:
        df = df.loc[df["gene"] == gene].copy()
    if gene is None and "gene" in df.columns and df["gene"].nunique() == 1:
        gene = str(df["gene"].iloc[0])

    if df.empty:
        return {"status": "skipped", "gene": gene, "reason": "no_rows_for_gene"}

    required = {"expr", "copies", "purity", "sf", subtype_col}
    missing = required - set(df.columns)
    if missing:
        return {"status": "error", "gene": gene, "reason": f"missing_columns: {sorted(missing)}"}

    # ---- CNA subset ----
    if cna == "amp":
        df = df[df["copies"] > (2 - et)]
    elif cna == "del":
        df = df[df["copies"] < (2 + et)]
    elif cna == "all":
        pass
    else:
        raise ValueError("cna must be 'amp', 'del', or 'all'")

    if df.empty:
        return {"status": "skipped", "gene": gene, "reason": "no_samples_after_cna_filter"}

    # ---- basic QC ----
    df = df.dropna(subset=list(required))
    if df.empty:
        return {"status": "skipped", "gene": gene, "reason": "all_na_after_dropna"}

    if (df["expr"] < 0).any():
        return {"status": "error", "gene": gene, "reason": "negative_counts"}

    if not df["purity"].between(0, 1).all():
        return {"status": "error", "gene": gene, "reason": "purity_out_of_bounds"}

    if not (df["sf"] > 0).all():
        return {"status": "error", "gene": gene, "reason": "nonpositive_sf"}

    if df["expr"].nunique() < min_unique_counts:
        return {"status": "skipped", "gene": gene, "reason": "too_few_unique_counts"}

    # aneuploid count check
    n_aneup = int((np.abs(df["copies"].astype(float) - 2.0) > (1.0 - et)).sum())
    if n_aneup < min_aneup or (df["expr"] == 0).all():
        return {
            "status": "skipped",
            "gene": gene,
            "n_aneup": n_aneup,
            "reason": "low_aneup_or_all_zero",
        }

    # identifiability check: need some CN deviation mass
    dev_tmp = (df["copies"].astype(float) - 2.0) / 2.0
    if cna == "all" and float(np.abs(dev_tmp).sum()) < min_cn_abs_sum:
        return {
            "status": "skipped",
            "gene": gene,
            "n_aneup": n_aneup,
            "reason": "too_little_cn_variation",
        }

    # ---- subtype encoding -----
    if subtype_order is not None:
        cat = pd.Categorical(df[subtype_col], categories=subtype_order, ordered=True)
        if cat.isna().any():
            bad = df.loc[cat.isna(), subtype_col].unique().tolist()
            return {
                "status": "error",
                "gene": gene,
                "reason": f"unknown_subtypes: {bad}",
            }
        subtype_codes = (cat.codes + 1).astype(int)
        levels = list(cat.categories)
    else:
        levels = sorted(pd.unique(df[subtype_col]).tolist())
        mapping = {lv: i + 1 for i, lv in enumerate(levels)}
        subtype_codes = df[subtype_col].map(mapping).astype(int).to_numpy()

    S = len(levels)
    if S < 2:
        return {
            "status": "skipped",
            "gene": gene,
            "reason": "need_at_least_2_subtypes_present_for_DE",
        }

    # ---- CN covariates (identified) ----
    # effective CN for scaling: treat 0 and 1 as "1 copy"
    CN_eff = df["copies"].astype(float).clip(lower=1.0)
    #df["dose_log"] = np.log(np.maximum(df["copies"].astype(float), 0.1) / 2.0)
    df["dose_log"] = np.log(CN_eff / 2.0)
    df["dev"] = (df["copies"].astype(float) - 2.0) / 2.0

    stan_data = {
        "N": int(len(df)),
        "y": df["expr"].astype(int).to_numpy(),
        "S": int(S),
        "subtype": np.asarray(subtype_codes, dtype=int),
        "sf": df["sf"].to_numpy(dtype=float),
        "purity": df["purity"].to_numpy(dtype=float),
        "dose_log": df["dose_log"].to_numpy(dtype=float),
        "dev": df["dev"].to_numpy(dtype=float),
    }

    # ---- initial values for NUTS ----
    rng = np.random.default_rng(seed)
    init_nuts = {
        # global means
        "b0_mean": float(rng.normal(0.0, 0.2)),
        "b_scaling_mean": float(rng.normal(0.0, 0.2)),
        "b_dev_mean": float(rng.normal(0.0, 0.05)),
        
        "b0_offset": rng.normal(0.0, 0.1, size=S).tolist(),
        "b_scaling_offset": rng.normal(0.0, 0.1, size=S).tolist(),
        "b_dev_offset": rng.normal(0.0, 0.05, size=S).tolist(),

        # non-centered subtype z's (std normal scale)
        #"z_b0": rng.normal(0.0, 0.1, size=S).tolist(),
        #"z_b_scaling": rng.normal(0.0, 0.1, size=S).tolist(),
        #"z_b_dev": rng.normal(0.0, 0.1, size=S).tolist(),

        # hierarchical SDs (must be positive)
        #"tau_b0": abs(rng.normal(0.1, 0.05)),
        #"tau_b_scaling": abs(rng.normal(0.1, 0.05)),
        #"tau_b_dev": abs(rng.normal(0.05, 0.02)),
        
        # stromal baseline
        "b_noncancer_log": float(rng.normal(0.0, 0.2)),
        
        # dispersion (now on log scale)
        "log_phi": float(rng.normal(0.0, 0.2)),
    }

    # ---- inference engine ----
    engine = engine.lower()
    if engine == "nuts":
        fit = model.sample(
            data=stan_data,
            chains=chains,
            iter_warmup=iter_warmup,
            iter_sampling=iter_sampling,
            seed=seed,
            inits=init_nuts,
            show_progress=show_progress,
            adapt_delta=adapt_delta,
            max_treedepth=max_treedepth,
        )
        draws = fit.draws_pd()
        using_mcmc = True

    elif engine in {"vi_meanfield", "vi_fullrank"}:
        algo = "meanfield" if engine == "vi_meanfield" else "fullrank"

        # NOTE: CmdStan's variational inits arg is a *float range*, not a dict.
        # We'll just let Stan pick its default random init unless you want to
        # pass a scalar, e.g. inits=0.1.
        fit = model.variational(
            data=stan_data,
            seed=seed,
            algorithm=algo,
            iter=vi_iter,
            grad_samples=vi_grad_samples,
            elbo_samples=vi_elbo_samples,
            output_samples=vi_output_samples,
            show_console=show_progress,
        )
        # VI: build a DataFrame manually
        # variational_sample: array of shape (draws, num_params)
        draws_np = fit.variational_sample
        colnames = fit.column_names    # e.g. ["b0_mean", "b0[1]", "b0[2]", ...]
        draws = pd.DataFrame(draws_np, columns=colnames)
        using_mcmc = False

    else:
        raise ValueError("engine must be 'nuts', 'vi_meanfield', or 'vi_fullrank'")

    # ---- helpers ----
    def q(x: np.ndarray) -> list[float]:
        return np.quantile(x, [0.025, 0.5, 0.975]).tolist()

    def summarize_draw(x: np.ndarray, prefix: str) -> dict:
        qi = q(x)
        return {
            f"{prefix}_mean": float(np.mean(x)),
            f"{prefix}_q025": float(qi[0]),
            f"{prefix}_q50": float(qi[1]),
            f"{prefix}_q975": float(qi[2]),
        }

    # ---- check essential columns ----
    required_cols = [
        "delta_tumor0_log",
        "delta_scaling",
        "delta_dev",
        "phi",
        "b_noncancer_log",
    ]
    missing_cols = [c for c in required_cols if c not in draws.columns]
    if missing_cols:
        return {
            "status": "error",
            "gene": gene,
            "reason": f"missing_draws_columns: {missing_cols}",
        }

    # ---- contrasts ----
    d_tumor = draws["delta_tumor0_log"].to_numpy()
    d_scal = draws["delta_scaling"].to_numpy()
    d_dev = draws["delta_dev"].to_numpy()

    # sign probabilities & lfsr
    p_up_tumor = float((d_tumor > 0).mean())
    lfsr_tumor = float(min(p_up_tumor, 1.0 - p_up_tumor))
    p_rope_tumor = float((np.abs(d_tumor) <= rope_logfc).mean())

    p_up_scal = float((d_scal > 0).mean())
    lfsr_scal = float(min(p_up_scal, 1.0 - p_up_scal))

    p_up_dev = float((d_dev > 0).mean())
    lfsr_dev = float(min(p_up_dev, 1.0 - p_up_dev))

    # log2 fold-change for tumor baseline DE
    ln2 = np.log(2.0)
    lfc_tumor = d_tumor / ln2
    lfc_ci = q(lfc_tumor)

    out: dict[str, object] = {
        "status": "ok",
        "gene": gene,
        "N": int(len(df)),
        "n_aneup": n_aneup,
        "cna": cna,
        "subtype_levels": levels,

        # tumor baseline DE at CN=2 (log2 scale)
        "tumor0_lfc_mean": float(np.mean(lfc_tumor)),
        "tumor0_lfc_q025": float(lfc_ci[0]),
        "tumor0_lfc_q975": float(lfc_ci[2]),
        "p_up_tumor": p_up_tumor,
        "lfsr_tumor": lfsr_tumor,
        "p_rope_tumor": p_rope_tumor,

        # differential CN wiring between subtypes
        "delta_scaling_mean": float(np.mean(d_scal)),
        "delta_scaling_q025": float(q(d_scal)[0]),
        "delta_scaling_q975": float(q(d_scal)[2]),
        "p_up_scaling": p_up_scal,
        "lfsr_scaling": lfsr_scal,

        "delta_dev_mean": float(np.mean(d_dev)),
        "delta_dev_q025": float(q(d_dev)[0]),
        "delta_dev_q975": float(q(d_dev)[2]),
        "p_up_dev": p_up_dev,
        "lfsr_dev": lfsr_dev,

        # dispersion
        "phi_mean": float(np.mean(draws["phi"].to_numpy())),
        "phi_q025": float(q(draws["phi"].to_numpy())[0]),
        "phi_q975": float(q(draws["phi"].to_numpy())[2]),
    }

    # ---- subtype-specific coefficients & dosage summaries -----
    s_iter = range(1, S + 1) if return_all_subtypes else range(1, min(S, 2) + 1)

    for s in s_iter:
        # coefficients b0, b_scaling, b_deviation
        for base in ["b0", "b_scaling", "b_deviation"]:
            col = f"{base}[{s}]"
            if col in draws.columns:
                arr = draws[col].to_numpy()
                out.update(summarize_draw(arr, f"{base}_s{s}"))

        # canonical CN transitions
        col21 = f"lp_2to1[{s}]"
        col23 = f"lp_2to3[{s}]"
        col24 = f"lp_2to4[{s}]"

        if col21 in draws.columns:
            lp21 = draws[col21].to_numpy()
            out.update(summarize_draw(lp21, f"lp_2to1_s{s}"))
            frac21 = np.expm1(lp21)
            out.update(summarize_draw(frac21, f"fracCN_2to1_s{s}"))
            out[f"p_fracCN_2to1_pos_s{s}"] = float((frac21 > eps_frac).mean())
            out[f"p_fracCN_2to1_rope_s{s}"] = float((np.abs(frac21) <= eps_frac).mean())
            out[f"p_fracCN_2to1_neg_s{s}"] = float((frac21 < -eps_frac).mean())

        if col23 in draws.columns:
            lp23 = draws[col23].to_numpy()
            out.update(summarize_draw(lp23, f"lp_2to3_s{s}"))
            frac23 = np.expm1(lp23)
            out.update(summarize_draw(frac23, f"fracCN_2to3_s{s}"))
            out[f"p_fracCN_2to3_pos_s{s}"] = float((frac23 > eps_frac).mean())
            out[f"p_fracCN_2to3_rope_s{s}"] = float((np.abs(frac23) <= eps_frac).mean())
            out[f"p_fracCN_2to3_neg_s{s}"] = float((frac23 < -eps_frac).mean())

        if col24 in draws.columns:
            lp24 = draws[col24].to_numpy()
            out.update(summarize_draw(lp24, f"lp_2to4_s{s}"))
            frac24 = np.expm1(lp24)
            out.update(summarize_draw(frac24, f"fracCN_2to4_s{s}"))
            out[f"p_fracCN_2to4_pos_s{s}"] = float((frac24 > eps_frac).mean())
            out[f"p_fracCN_2to4_rope_s{s}"] = float((np.abs(frac24) <= eps_frac).mean())
            out[f"p_fracCN_2to4_neg_s{s}"] = float((frac24 < -eps_frac).mean())

        # Optional: if you added cancel_index_2to*, p_DC_* in Stan, they will be
        # present in draws and you can either pick them up here or classify later
        # from lp_scaling_* and lp_dev_*.

        # Also summarize lp_scaling_* and lp_dev_* if present (useful for diagnostics)
        for trans in ["2to1", "2to3", "2to4"]:
            for comp in ["scaling", "dev"]:
                col = f"lp_{comp}_{trans}[{s}]"
                if col in draws.columns:
                    arr = draws[col].to_numpy()
                    out.update(summarize_draw(arr, f"lp_{comp}_{trans}_s{s}"))

    # ---- Sampler diagnostics (for model checking) ----
    if using_mcmc:
        summ = fit.summary()
        rhat_col = next((c for c in ["R_hat", "Rhat"] if c in summ.columns), None)
        ess_col = next((c for c in ["Ess_bulk", "ESS_bulk", "N_Eff", "Ess"] if c in summ.columns), None)

        # Rhat / ESS for phi 
        if rhat_col and "phi" in summ.index:
            out["Rhat_phi"] = float(summ.loc["phi", rhat_col])
        if ess_col and "phi" in summ.index:
            out["ess_phi"] = float(summ.loc["phi", ess_col])

        # Rhat / ESS over a core set of parameters (for global diagnostics)
        core_params = ["delta_tumor0_log", "delta_scaling", "delta_dev"]
        for s in s_iter:
            core_params += [f"b0[{s}]", f"b_scaling[{s}]", f"b_deviation[{s}]"]

        core_params = [p for p in core_params if p in summ.index]

        if core_params and rhat_col and ess_col:
            rhat_vals = summ.loc[core_params, rhat_col].to_numpy(dtype=float)
            ess_vals = summ.loc[core_params, ess_col].to_numpy(dtype=float)
            out["max_Rhat_core"] = float(np.nanmax(rhat_vals))
            out["min_ESS_core"] = float(np.nanmin(ess_vals))
        else:
            out["max_Rhat_core"] = np.nan
            out["min_ESS_core"] = np.nan

        # Divergences and treedepth from sampler params (if present)
        if "divergent__" in draws.columns:
            out["n_divergent"] = int(draws["divergent__"].sum())
        if "treedepth__" in draws.columns:
            td = draws["treedepth__"].to_numpy()
            out["max_treedepth"] = int(td.max())
            out["n_max_treedepth"] = int((td >= max_treedepth).sum())

        # simple flag based on diagnostics
        out["fit_flag"] = "ok"
        if (
            ("max_Rhat_core" in out and out["max_Rhat_core"] is not np.nan and out["max_Rhat_core"] > 1.05)
            or ("min_ESS_core" in out and out["min_ESS_core"] is not np.nan and out["min_ESS_core"] < 100)
            or ("n_divergent" in out and out["n_divergent"] > 0)
        ):
            out["fit_flag"] = "warn"
    else:
        # VI: no HMC diagnostics; you can add ELBO here if you want
        out["Rhat_phi"] = np.nan
        out["ess_phi"] = np.nan
        out["max_Rhat_core"] = np.nan
        out["min_ESS_core"] = np.nan
        out["n_divergent"] = 0
        out["max_treedepth"] = 0
        out["n_max_treedepth"] = 0
        out["fit_flag"] = "ok"

    return out

In [25]:
# Fit on simulated data

def run_gene(gene_id, data_df,seed_base=1):
    """
    Fit one gene and return the result dict from fit_one_gene_de.
    """
    df_g = data_df[data_df["gene"] == gene_id].copy()

    # Optional: give each gene a different seed for reproducibility
    # (and so chains are different across genes)
    seed = seed_base + hash(gene_id) % 10_000_000

    res = fit_one_gene_de(
        df_g,
        model,
        gene=gene_id,
        engine="nuts",
        chains=4,
        iter_warmup=1000,
        iter_sampling=1000,
        seed=seed,
        show_progress=False,
        adapt_delta=0.9,
        max_treedepth=12,
    )
    return res

genes = sim_df["gene"].unique()

results = Parallel(n_jobs=4, backend="loky")(
    delayed(run_gene)(g, sim_df) for g in genes
)

results_df = pd.DataFrame(results)

15:22:54 - cmdstanpy - INFO - CmdStan start processing
15:22:54 - cmdstanpy - INFO - CmdStan start processing
15:22:54 - cmdstanpy - INFO - CmdStan start processing
15:22:54 - cmdstanpy - INFO - Chain [1] start processing
15:22:54 - cmdstanpy - INFO - Chain [2] start processing
15:22:54 - cmdstanpy - INFO - Chain [3] start processing
15:22:54 - cmdstanpy - INFO - Chain [4] start processing
15:22:54 - cmdstanpy - INFO - Chain [2] start processing
15:22:54 - cmdstanpy - INFO - Chain [3] start processing
15:22:54 - cmdstanpy - INFO - Chain [2] start processing
15:22:54 - cmdstanpy - INFO - Chain [1] start processing
15:22:54 - cmdstanpy - INFO - Chain [1] start processing
15:22:54 - cmdstanpy - INFO - Chain [4] start processing
15:22:54 - cmdstanpy - INFO - Chain [3] start processing
15:22:54 - cmdstanpy - INFO - Chain [4] start processing
15:46:16 - cmdstanpy - INFO - Chain [2] done processing
15:46:22 - cmdstanpy - INFO - Chain [1] done processing
15:46:22 - cmdstanpy - INFO - Chain [1]

In [1]:
#res_df = pd.DataFrame([res_mcmc])
#res_df.head()

In [15]:
# VI with mean-field Gaussian (fast)
res_vi = fit_one_gene_de(
    gene_df, 
    model,
    engine="vi_meanfield",
    vi_iter=20000,
    vi_output_samples=2000
)

22:53:33 - cmdstanpy - WARNING - Argument name `output_samples` is deprecated, please rename to `draws`.
22:53:33 - cmdstanpy - INFO - Chain [1] start processing
22:53:35 - cmdstanpy - INFO - Chain [1] done processing


In [17]:
res_vi

{'status': 'ok',
 'gene': 'KRAS',
 'N': 862,
 'n_aneup': 67,
 'cna': 'all',
 'subtype_levels': ['MSI', 'MSS'],
 'tumor0_lfc_mean': -0.043490314574413776,
 'tumor0_lfc_q025': -0.16337258979906732,
 'tumor0_lfc_q975': 0.07362380087700086,
 'p_up_tumor': 0.2375,
 'lfsr_tumor': 0.2375,
 'p_rope_tumor': 1.0,
 'delta_scaling_mean': -0.2151235339914,
 'delta_scaling_q025': -0.562249275,
 'delta_scaling_q975': 0.13101847499999966,
 'p_up_scaling': 0.11,
 'lfsr_scaling': 0.11,
 'delta_dev_mean': 0.0543051180345,
 'delta_dev_q025': -0.13608615000000002,
 'delta_dev_q975': 0.23973734999999993,
 'p_up_dev': 0.717,
 'lfsr_dev': 0.28300000000000003,
 'phi_mean': 17.959554649999998,
 'phi_q025': 16.2621725,
 'phi_q975': 19.840419999999998,
 'b0_s1_mean': 7.650280465,
 'b0_s1_q025': 7.60378725,
 'b0_s1_q50': 7.650295,
 'b0_s1_q975': 7.69738425,
 'b_scaling_s1_mean': 1.272089134,
 'b_scaling_s1_q025': 1.029407,
 'b_scaling_s1_q50': 1.2713700000000001,
 'b_scaling_s1_q975': 1.5093717499999997,
 'b_devia

#### Parallel run on multiple genes

In [19]:
gene_groups = {g: gdf for g, gdf in df_long.groupby("gene", sort=False)}

def run_gene(g):
    return fit_one_gene_de(
        gene_groups[g],
        model,
        cna="all",
        chains=4,
        iter_warmup=500,
        iter_sampling=500,
        adapt_delta=0.90
    )

genes = list(gene_groups.keys())

results = Parallel(n_jobs=8, backend="loky")(
    delayed(run_gene)(g) for g in genes
)

In [55]:
res_df = pd.DataFrame(results)
res_df.head()

,status,gene,N,n_aneup,cna,subtype_levels,tumor0_lfc_mean,tumor0_lfc_q025,tumor0_lfc_q975,p_up_tumor,...,p_lp_dev_2to4_neg_s2,cancel_index_2to4_s2_mean,cancel_index_2to4_s2_q025,cancel_index_2to4_s2_q50,cancel_index_2to4_s2_q975,p_DC_gain_s2,p_DC_loss_s2,Rhat_phi,ess_phi,fit_flag
0,ok,BRAF,986,139,all,"[MSI, MSS]",0.515110,0.417934,0.612417,1.000,...,0.0730,0.170276,-0.197589,0.128018,0.784141,0.0,0.0,0.999300,2299.740496,ok
1,ok,KRAS,986,69,all,"[MSI, MSS]",-0.015753,-0.122581,0.092460,0.386,...,0.0605,-0.012394,-0.102862,-0.017951,0.106385,0.0,0.0,0.999700,2072.642570,ok
2,ok,TP53,986,217,all,"[MSI, MSS]",-0.018923,-0.176240,0.130874,0.405,...,0.1765,0.063260,-0.351281,0.037346,0.659321,0.0,0.0,2.528062,2.346820,warn


#### Downstream results interpretation

In [95]:
@dataclass
class InterpretThresholds:
    # DE thresholds 
    de_lfsr_sig: float = 0.05        # call DE if lfsr <= this
    de_rope_high: float = 0.90       # call DE-null if p_rope >= this

    # Rewiring thresholds 
    rewire_lfsr_sig: float = 0.10    # call rewiring if lfsr <= this

    # --- Dosage class thresholds (per subtype) ---
    # We use posterior probabilities that our fit fn already computes.
    # "Sensitive" means CN perturbation yields effect beyond ROPE with high prob.
    dose_prob_sens: float = 0.80     # e.g. p_fracCN_2to3_pos >= 0.90 => sensitive to gain
    dose_prob_ins: float = 0.80      # e.g. p_fracCN_2to3_rope >= 0.90 => insensitive to gain

    # Dosage-compensation thresholds (if you output p_DC_gain_sX / p_DC_loss_sX)
    dc_prob: float = 0.80            # call compensated if >= this

    # If p_DC_* are not present, you can fall back to "cancel_index" if you output it:
    cancel_abs_rope: float = 0.20    # |cancel_index| <= this => strong cancellation (optional)

def _get(res: Dict[str, Any], key: str, default=None):
    return res.get(key, default)

def interpret_gene_result(
    res: Dict[str, Any],
    th: InterpretThresholds = InterpretThresholds(),
    assume_pairwise_s2_vs_s1: bool = True,
) -> Dict[str, Any]:
    """
    Interpret one gene's fitted result dict:
      - DE status (tumor baseline, subtype2 vs subtype1)
      - rewiring status (scaling / deviation)
      - dosage class per subtype (DSG / DIG / DCG) with optional gain/loss flags

    Returns a FLAT dict (good for DataFrame rows).
    """

    out: Dict[str, Any] = {
        "gene": _get(res, "gene"),
        "status": _get(res, "status"),
        "fit_flag": _get(res, "fit_flag", "ok"),
        "N": _get(res, "N"),
        "n_aneup": _get(res, "n_aneup"),
    }


    # 1) DE status (tumor baseline)

    lfsr_tumor = _get(res, "lfsr_tumor", np.nan)
    p_rope_tumor = _get(res, "p_rope_tumor", np.nan)

    if np.isfinite(lfsr_tumor) and lfsr_tumor <= th.de_lfsr_sig:
        de_status = "DE"
    elif np.isfinite(p_rope_tumor) and p_rope_tumor >= th.de_rope_high:
        de_status = "DE-null"
    else:
        de_status = "DE-uncertain"

    out.update({
        "de_status": de_status,
        "lfsr_tumor": lfsr_tumor,
        "p_rope_tumor": p_rope_tumor,
        # keep your effect size if present
        "tumor0_lfc_mean": _get(res, "tumor0_lfc_mean", np.nan),
        "tumor0_lfc_q025": _get(res, "tumor0_lfc_q025", np.nan),
        "tumor0_lfc_q975": _get(res, "tumor0_lfc_q975", np.nan),
    })

  
    # 2) Rewiring status (between subtypes)
    
    lfsr_scaling = _get(res, "lfsr_scaling", np.nan)
    lfsr_dev = _get(res, "lfsr_dev", np.nan)

    scaling_rewired = (np.isfinite(lfsr_scaling) and lfsr_scaling <= th.rewire_lfsr_sig)
    dev_rewired     = (np.isfinite(lfsr_dev) and lfsr_dev <= th.rewire_lfsr_sig)

    if scaling_rewired and dev_rewired:
        rewiring = "rewired:scaling+deviation"
    elif scaling_rewired:
        rewiring = "rewired:scaling"
    elif dev_rewired:
        rewiring = "rewired:deviation"
    else:
        rewiring = "not_rewired"

    out.update({
        "rewiring_status": rewiring,
        "lfsr_scaling": lfsr_scaling,
        "lfsr_dev": lfsr_dev,
        "delta_scaling_mean": _get(res, "delta_scaling_mean", np.nan),
        "delta_scaling_q025": _get(res, "delta_scaling_q025", np.nan),
        "delta_scaling_q975": _get(res, "delta_scaling_q975", np.nan),
        "delta_dev_mean": _get(res, "delta_dev_mean", np.nan),
        "delta_dev_q025": _get(res, "delta_dev_q025", np.nan),
        "delta_dev_q975": _get(res, "delta_dev_q975", np.nan),
    })

    
    # 3) Dosage class per subtype
    
    subtype_levels = _get(res, "subtype_levels", None)  # e.g. ["MSI","MSS"]
    if subtype_levels is None:
        # still proceed with s1,s2,... keys
        subtype_levels = []

    # infer how many subtypes are present from outputs (robust)
    # Prefer explicit list; else scan for p_fracCN_2to3_* keys
    S = len(subtype_levels)
    if S == 0:
        # detect max s index present
        s_candidates = []
        for k in res.keys():
            if k.startswith("p_fracCN_2to3_pos_s"):
                try:
                    s_candidates.append(int(k.split("_s")[-1]))
                except Exception:
                    pass
        S = max(s_candidates) if s_candidates else 2  # fallback

    out["S"] = S
    if subtype_levels:
        out["subtype_levels_str"] = "|".join(map(str, subtype_levels))

    def subtype_name(s: int) -> str:
        if 1 <= s <= len(subtype_levels):
            return str(subtype_levels[s-1])
        return f"s{s}"

    # helper for dosage decision per subtype
    def dosage_class_for_subtype(s: int) -> Dict[str, Any]:
        name = subtype_name(s)

        # Evidence for being sensitive vs insensitive for gain (2->3) and loss (2->1)
        p_gain_pos  = _get(res, f"p_fracCN_2to3_pos_s{s}", np.nan)
        p_gain_rope = _get(res, f"p_fracCN_2to3_rope_s{s}", np.nan)
        p_loss_neg  = _get(res, f"p_fracCN_2to1_neg_s{s}", np.nan)
        p_loss_rope = _get(res, f"p_fracCN_2to1_rope_s{s}", np.nan)

        # Optional DC outputs
        p_dc_gain = _get(res, f"p_DC_gain_s{s}", np.nan)
        p_dc_loss = _get(res, f"p_DC_loss_s{s}", np.nan)

        # Optional cancellation index (if you output it)
        cancel_gain_mean = _get(res, f"cancel_index_2to3_s{s}_mean", np.nan)
        cancel_loss_mean = _get(res, f"cancel_index_2to1_s{s}_mean", np.nan)

        # classify gain/loss directionally (useful diagnostics)
        gain_flag = "unknown"
        if np.isfinite(p_gain_pos) and p_gain_pos >= th.dose_prob_sens:
            gain_flag = "sensitive"
        elif np.isfinite(p_gain_rope) and p_gain_rope >= th.dose_prob_ins:
            gain_flag = "insensitive"

        loss_flag = "unknown"
        if np.isfinite(p_loss_neg) and p_loss_neg >= th.dose_prob_sens:
            loss_flag = "sensitive"
        elif np.isfinite(p_loss_rope) and p_loss_rope >= th.dose_prob_ins:
            loss_flag = "insensitive"

        # primary subtype dosage class
        # Priority: DC > DIS > DS > uncertain
        dc_gain = (np.isfinite(p_dc_gain) and p_dc_gain >= th.dc_prob)
        dc_loss = (np.isfinite(p_dc_loss) and p_dc_loss >= th.dc_prob)

        # Fallback DC using cancellation index if p_DC_* missing
        if (not np.isfinite(p_dc_gain)) and np.isfinite(cancel_gain_mean):
            dc_gain = (abs(cancel_gain_mean) <= th.cancel_abs_rope)
        if (not np.isfinite(p_dc_loss)) and np.isfinite(cancel_loss_mean):
            dc_loss = (abs(cancel_loss_mean) <= th.cancel_abs_rope)

        any_dc = dc_gain or dc_loss

        # DIS if both gain and loss are ROPE-mostly (or at least gain ROPE-mostly and loss ROPE-mostly)
        dis_gain = (np.isfinite(p_gain_rope) and p_gain_rope >= th.dose_prob_ins)
        dis_loss = (np.isfinite(p_loss_rope) and p_loss_rope >= th.dose_prob_ins)
        is_dis = dis_gain and dis_loss

        # DS if strong sensitivity at least on one side (commonly gain), ideally both
        ds_gain = (np.isfinite(p_gain_pos) and p_gain_pos >= th.dose_prob_sens)
        ds_loss = (np.isfinite(p_loss_neg) and p_loss_neg >= th.dose_prob_sens)
        is_ds = ds_gain or ds_loss

        if any_dc:
            cls = "DCG"   # dosage-compensated
        elif is_dis:
            cls = "DIG"  # dosage-insensitive
        elif is_ds:
            cls = "DSG"   # dosage-sensitive
        else:
            cls = "UNC"

        # attach a few useful summary numbers if present
        return {
            f"dosage_class_{name}": cls,
            f"gain_flag_{name}": gain_flag,
            f"loss_flag_{name}": loss_flag,
            f"p_gain_pos_{name}": p_gain_pos,
            f"p_gain_rope_{name}": p_gain_rope,
            f"p_loss_neg_{name}": p_loss_neg,
            f"p_loss_rope_{name}": p_loss_rope,
            f"p_DC_gain_{name}": p_dc_gain,
            f"p_DC_loss_{name}": p_dc_loss,
        }

    for s in range(1, S + 1):
        out.update(dosage_class_for_subtype(s))

    # 4) Optional combined label for easy plotting
    
    # Example: "DE-null | not_rewired | MSI:DS, MSS:DS"
    per_sub = []
    for s in range(1, S + 1):
        name = subtype_name(s)
        per_sub.append(f"{name}:{out.get(f'dosage_class_{name}', 'NA')}")
    out["summary_label"] = f"{de_status} | {rewiring} | " + ",".join(per_sub)

    return out

In [13]:
# Load CRC driver fit results

DATA_PATH = "/Users/katsiarynadavydzenka/Documents/PhD_AI/CRC_case_study/results/"
res_drivers = pd.read_csv(os.path.join(DATA_PATH, "crc_res_nb_de_drivers.csv"))
res_drivers.head()

,Unnamed: 0.1,Unnamed: 0,gene,success,error,status,gene.1,N,n_aneup,cna,...,lp_dev_2to4_s2_q50,lp_dev_2to4_s2_q975,Rhat_phi,ess_phi,max_Rhat_core,min_ESS_core,n_divergent,max_treedepth,n_max_treedepth,fit_flag
0,0,0,ACVR1B,True,NaN,ok,ACVR1B,986,39,all,...,0.021568,0.256968,1.00153,4073.18,1.00227,2950.73,0,6,0,ok
1,1,1,ACVR2A,True,NaN,ok,ACVR2A,986,18,all,...,0.049770,0.290543,1.00023,4619.61,1.00232,2701.66,0,7,0,ok
2,2,2,AKT1,True,NaN,ok,AKT1,986,95,all,...,0.103485,0.330054,1.00009,5740.17,1.00137,3534.62,0,6,0,ok
3,3,3,AMER1,True,NaN,ok,AMER1,986,173,all,...,0.242166,0.451338,1.00030,4601.64,1.00370,3454.74,0,6,0,ok
4,4,4,ANKRD40,True,NaN,ok,ANKRD40,986,50,all,...,0.113387,0.306108,1.00208,4200.89,1.00086,3346.70,0,6,0,ok


In [15]:
res_drivers = res_drivers.drop(
    columns=['Unnamed: 0.1', 'Unnamed: 0', 'gene.1', 'success', 'error']
)
res_drivers.head()

,gene,status,N,n_aneup,cna,subtype_levels,tumor0_lfc_mean,tumor0_lfc_q025,tumor0_lfc_q975,p_up_tumor,...,lp_dev_2to4_s2_q50,lp_dev_2to4_s2_q975,Rhat_phi,ess_phi,max_Rhat_core,min_ESS_core,n_divergent,max_treedepth,n_max_treedepth,fit_flag
0,ACVR1B,ok,986,39,all,"['MSI', 'MSS']",0.986634,0.848520,1.132959,1.00000,...,0.021568,0.256968,1.00153,4073.18,1.00227,2950.73,0,6,0,ok
1,ACVR2A,ok,986,18,all,"['MSI', 'MSS']",0.153927,0.018246,0.287838,0.98750,...,0.049770,0.290543,1.00023,4619.61,1.00232,2701.66,0,7,0,ok
2,AKT1,ok,986,95,all,"['MSI', 'MSS']",-0.386074,-0.465721,-0.304330,0.00000,...,0.103485,0.330054,1.00009,5740.17,1.00137,3534.62,0,6,0,ok
3,AMER1,ok,986,173,all,"['MSI', 'MSS']",0.910547,0.756041,1.070976,1.00000,...,0.242166,0.451338,1.00030,4601.64,1.00370,3454.74,0,6,0,ok
4,ANKRD40,ok,986,50,all,"['MSI', 'MSS']",0.139085,0.040859,0.235130,0.99725,...,0.113387,0.306108,1.00208,4200.89,1.00086,3346.70,0,6,0,ok


In [7]:
type(res_drivers.loc[0, "result"])

str

In [9]:
res_drivers["result"] = res_drivers["result"].apply(ast.literal_eval)
result_df = pd.json_normalize(res_drivers["result"])
df_expanded = pd.concat([res_drivers.drop(columns=["result"]), result_df], axis=1)
df_expanded.head()

,Unnamed: 0,gene,success,error,status,gene,N,n_aneup,cna,subtype_levels,...,lp_dev_2to4_s2_q50,lp_dev_2to4_s2_q975,Rhat_phi,ess_phi,max_Rhat_core,min_ESS_core,n_divergent,max_treedepth,n_max_treedepth,fit_flag
0,0,ACVR1B,True,NaN,ok,ACVR1B,986,39,all,"[MSI, MSS]",...,0.021568,0.256968,1.00153,4073.18,1.00227,2950.73,0,6,0,ok
1,1,ACVR2A,True,NaN,ok,ACVR2A,986,18,all,"[MSI, MSS]",...,0.049770,0.290543,1.00023,4619.61,1.00232,2701.66,0,7,0,ok
2,2,AKT1,True,NaN,ok,AKT1,986,95,all,"[MSI, MSS]",...,0.103485,0.330054,1.00009,5740.17,1.00137,3534.62,0,6,0,ok
3,3,AMER1,True,NaN,ok,AMER1,986,173,all,"[MSI, MSS]",...,0.242166,0.451338,1.00030,4601.64,1.00370,3454.74,0,6,0,ok
4,4,ANKRD40,True,NaN,ok,ANKRD40,986,50,all,"[MSI, MSS]",...,0.113387,0.306108,1.00208,4200.89,1.00086,3346.70,0,6,0,ok


In [17]:
res_drivers.to_csv("/Users/katsiarynadavydzenka/Documents/PhD_AI/CRC_case_study/results/crc_res_nb_de_drivers.csv", index=True)

In [105]:
filtered = res_drivers[res_drivers["fit_flag"] == "ok"]

interpreter = [
    interpret_gene_result(r)
    for r in filtered.to_dict("records")
]